# Stroke Detection Model

## Loading Necessary Libraries and Packages

In [1]:
import pandas as pd

## Loading, Inspecting and Cleaning the Data

In [2]:
df = pd.read_csv("/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/data/healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
df.info()
df.describe()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

### Median Imputation for missing BMI values

In [4]:
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

### Dropping ID Column

In [5]:
df = df.drop('id', axis=1)

### One Hot Encoding Categorical Columns

In [6]:
categorical_cols = ['gender', 'ever_married', 'work_type',
                    'Residence_type', 'smoking_status']

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [7]:
df.to_csv("cleaned_stroke_dataset.csv", index=False)

### Showing Cleaned Dataset

In [8]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   age                             5110 non-null   float64
 1   hypertension                    5110 non-null   int64  
 2   heart_disease                   5110 non-null   int64  
 3   avg_glucose_level               5110 non-null   float64
 4   bmi                             5110 non-null   float64
 5   stroke                          5110 non-null   int64  
 6   gender_Male                     5110 non-null   bool   
 7   gender_Other                    5110 non-null   bool   
 8   ever_married_Yes                5110 non-null   bool   
 9   work_type_Never_worked          5110 non-null   bool   
 10  work_type_Private               5110 non-null   bool   
 11  work_type_Self-employed         5110 non-null   bool   
 12  work_type_children              51

# Building and Comparing Models
## Baseline Models


### 1. Random Forest Classifier

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix


# 1. Load Data

df = pd.read_csv("/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/data/cleaned_stroke_dataset.csv")

# Separate features and target
X = df.drop("stroke", axis=1)
y = df["stroke"]

# Train-test split (stratified due to class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# 2. Helper Function

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    # Handle models that do NOT have predict_proba (e.g., LinearSVC)
    if hasattr(model, "predict_proba"):
        y_scores = model.predict_proba(X_test)[:, 1]
    else:
        # Use decision_function instead, then normalize to [0,1]
        decision = model.decision_function(X_test)
        # Min-max scale to convert to probability-like scores
        y_scores = (decision - decision.min()) / (decision.max() - decision.min())

    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("ROC–AUC:", roc_auc_score(y_test, y_scores))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


# 3. Baseline Random Forest Model

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=42
)

# Train the model
rf.fit(X_train, y_train)


# 4. Evaluate

print("=== Random Forest Baseline  ===")
evaluate_model(rf, X_test, y_test)

=== Random Forest Baseline  ===
Classification Report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022

ROC-AUC: 0.8042798353909465
Confusion Matrix:
 [[971   1]
 [ 50   0]]


- The baseline Random Forest classifier achieved a high overall accuracy (95%); however, this result is misleading. 
* The model completely failed to identify any stroke cases, as reflected by a recall of 0.00 for the minority class. 
- The confusion matrix confirms that all 50 stroke instances in the test set were misclassified as non-stroke. This occurred due to the severe class imbalance in the dataset, where stroke cases represent less than 5% of samples. 
- These findings show that the baseline model does not meet the clinical objective of detecting stroke risk and highlight the necessity for more advanced imbalance-handling methods such as SMOTE, class weighting, and probability threshold adjustment.”

### 2. Logistic Regression

In [10]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression Baseline

log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model
log_reg.fit(X_train, y_train)

# Evaluate
print("=== Logistic Regression Baseline (No SMOTE, No Class Weights) ===")
evaluate_model(log_reg, X_test, y_test)

=== Logistic Regression Baseline (No SMOTE, No Class Weights) ===
Classification Report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.98       972
           1       1.00      0.02      0.04        50

    accuracy                           0.95      1022
   macro avg       0.98      0.51      0.51      1022
weighted avg       0.95      0.95      0.93      1022

ROC-AUC: 0.842201646090535
Confusion Matrix:
 [[972   0]
 [ 49   1]]


The baseline Logistic Regression model achieved a high accuracy of 95%, but this number is misleading due to the class imbalance in the dataset. While the model correctly classified almost all non-stroke cases, it detected only 11 out of 50 actual stroke cases, resulting in a very low recall of 0.02 for the stroke class.

This means that the model is predicting “no stroke” for nearly everyone, which makes the model unreliable for medical decision-making. However, the ROC–AUC score of 0.84 shows that the model can still separate high-risk and low-risk patients reasonably well at the probability level. The problem is that, with the default threshold of 0.5, the model does not convert enough of these higher-risk probabilities into positive predictions.

Overall, these results show that the Logistic Regression model, in its baseline form, is not able to identify stroke cases effectively. This confirms the need to apply techniques such as class weighting, SMOTE, or threshold adjustment to improve detection of the minority class.

### 3. Support Vector Machine

In [11]:
from sklearn.svm import SVC

svm_clf = SVC(kernel='linear', probability=True, random_state=42)
svm_clf.fit(X_train, y_train)

print("=== SVM Linear Baseline ===")
evaluate_model(svm_clf, X_test, y_test)

=== SVM Linear Baseline ===
Classification Report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022

ROC-AUC: 0.7389917695473252
Confusion Matrix:
 [[972   0]
 [ 50   0]]


/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in lab

The baseline SVM model achieved high accuracy (95%) but completely failed to detect any stroke cases, resulting in a recall of 0.00 for the stroke class. This means the model predicted “no stroke” for every patient. Although this produces high accuracy due to the dataset being highly imbalanced, it makes the model ineffective for medical prediction, where identifying positive cases is the main priority.

The ROC–AUC score of 0.74 indicates that the model has some ability to separate stroke from non-stroke cases at the probability level, but this was not reflected in the final predictions. SVMs also perform poorly on imbalanced data when the features are not standardized, which contributes to this outcome.

Overall, the baseline SVM model does not provide useful predictions for the stroke class. This reinforces the need to apply methods such as class weighting, feature scaling, and SMOTE to improve its performance.


## Weighted Models

### 1. Weighted Random Forest Classifier

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf_balanced = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)

rf_balanced.fit(X_train, y_train)

print("=== Random Forest (Balanced Class Weight) ===")
evaluate_model(rf_balanced, X_test, y_test)

=== Random Forest (Balanced Class Weight) ===
Classification Report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022

ROC-AUC: 0.8042798353909465
Confusion Matrix:
 [[971   1]
 [ 50   0]]


### 2. Weighted Logistic Regression

In [13]:
from sklearn.linear_model import LogisticRegression

log_reg_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

log_reg_balanced.fit(X_train, y_train)

print("=== Logistic Regression (Balanced Class Weight) ===")
evaluate_model(log_reg_balanced, X_test, y_test)

=== Logistic Regression (Balanced Class Weight) ===
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.74      0.85       972
           1       0.14      0.80      0.24        50

    accuracy                           0.75      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.75      0.82      1022

ROC-AUC: 0.8437242798353909
Confusion Matrix:
 [[723 249]
 [ 10  40]]


### 3. Weighted Suport Vector Machines Model

In [14]:
from sklearn.svm import SVC

svm_balanced = SVC(
    kernel='linear',
    probability=True,
    class_weight='balanced',
    random_state=42
)

svm_balanced.fit(X_train, y_train)

print("=== SVM Linear (Balanced Class Weight) ===")
evaluate_model(svm_balanced, X_test, y_test)

=== SVM Linear (Balanced Class Weight) ===
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.73      0.84       972
           1       0.13      0.80      0.23        50

    accuracy                           0.73      1022
   macro avg       0.56      0.76      0.53      1022
weighted avg       0.94      0.73      0.81      1022

ROC-AUC: 0.8411316872427985
Confusion Matrix:
 [[707 265]
 [ 10  40]]



### Model Performance Comparison (Baseline vs Class Weight Models)

| Model | Recall (Stroke) | Precision (Stroke) | Accuracy | ROC–AUC | Notes |
|-------|------------------|---------------------|----------|---------|-------|
| Random Forest (Baseline) | 0.00 | 0.00 | 0.95 | 0.80 | Predicted all cases as non-stroke |
| Logistic Regression (Baseline) | 0.02 | 1.00 | 0.95 | 0.84 | Very low recall; high AUC |
| SVM Linear (Baseline) | 0.00 | 0.00 | 0.95 | 0.74 | Failed to detect any stroke cases |
| Logistic Regression (Balanced) | **0.80** | 0.14 | 0.75 | 0.84 | Huge improvement in recall |
| SVM Linear (Balanced) | **0.80** | 0.13 | 0.73 | 0.84 | Strong recall, lower precision |
```

Applying class weights greatly improved the model’s ability to detect stroke cases. Both Logistic Regression and the Linear SVM increased their recall for the stroke class from almost zero in the baseline models to 0.80, meaning they correctly identified 40 out of 50 stroke cases. This is a major improvement and shows that class weighting helps the model focus more on the minority class.

However, this improvement comes with trade-offs. The precision for the stroke class dropped to around 0.13–0.14, meaning the models produced many false positives. The overall accuracy also decreased to about 73–75%, which is expected because the models are now predicting more “stroke” cases.

Despite these trade-offs, these results are important because in medical problems it is usually more important to identify as many true stroke cases as possible (high recall), even if it means allowing more false positives. The ROC–AUC values remained strong (around 0.84), showing that the models still have good ability to separate stroke from non-stroke cases based on predicted probabilities.

These results confirm that class weighting is helpful, but additional techniques like SMOTE and threshold tuning will still be needed to achieve a better balance between recall and precision.

## SMOTE Models

### Importing Necesary Libraries and Packages and Creating SMOTE Pipeline

In [15]:
import imblearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

### Random Forest + Smote

In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_smote = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

rf_smote.fit(X_train, y_train)

print("=== Random Forest + SMOTE ===")
evaluate_model(rf_smote, X_test, y_test)

=== Random Forest + SMOTE ===
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.96      0.96       972
           1       0.17      0.16      0.16        50

    accuracy                           0.92      1022
   macro avg       0.56      0.56      0.56      1022
weighted avg       0.92      0.92      0.92      1022

ROC-AUC: 0.774537037037037
Confusion Matrix:
 [[932  40]
 [ 42   8]]


### Logistic Regression + SMOTE

In [17]:
from sklearn.linear_model import LogisticRegression

log_reg_smote = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_smote.fit(X_train, y_train)

print("=== Logistic Regression + SMOTE ===")
evaluate_model(log_reg_smote, X_test, y_test)

=== Logistic Regression + SMOTE ===
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.87      0.92       972
           1       0.16      0.48      0.24        50

    accuracy                           0.85      1022
   macro avg       0.56      0.67      0.58      1022
weighted avg       0.93      0.85      0.88      1022

ROC-AUC: 0.7869341563786008
Confusion Matrix:
 [[844 128]
 [ 26  24]]


### Linear Support Vector Machine + SMOTE

In [18]:
from sklearn.svm import SVC

svm_smote = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', SVC(kernel='linear', probability=True, random_state=42))
])

svm_smote.fit(X_train, y_train)

print("=== SVM Linear + SMOTE ===")
evaluate_model(svm_smote, X_test, y_test)

=== SVM Linear + SMOTE ===
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.87      0.92       972
           1       0.15      0.44      0.22        50

    accuracy                           0.85      1022
   macro avg       0.56      0.66      0.57      1022
weighted avg       0.93      0.85      0.88      1022

ROC-AUC: 0.7741975308641975
Confusion Matrix:
 [[847 125]
 [ 28  22]]


| Model                       | Recall (Class 1) | Precision (Class 1) | F1 (Class 1) | ROC-AUC | Notes |
|-----------------------------|------------------|-----------------------|--------------|---------|-------|
| Random Forest + SMOTE      | 0.16             | 0.17                  | 0.16         | 0.77    | Poor minority detection |
| Logistic Regression + SMOTE| 0.48             | 0.16                  | 0.24         | 0.79    | Best recall so far |
| SVM Linear + SMOTE         | 0.44             | 0.15                  | 0.22         | 0.77    | Similar to LR but slightly lower recall |

After applying SMOTE to oversample the minority (stroke) class, we trained three models: Random Forest, Logistic Regression, and a Linear SVM. The goal was to see whether balancing the training data would improve the detection of stroke cases compared to the previous class-weighted models.

**Random Forest + SMOTE**  
The Random Forest model achieved high overall accuracy (92%) and very good performance on the non-stroke class. However, it still performed poorly on the stroke class, with a recall of only 0.16 (8 out of 50 stroke cases correctly identified). This shows that even after SMOTE, Random Forest remains biased toward the majority class and is not suitable on its own for stroke detection.

**Logistic Regression + SMOTE**  
For Logistic Regression, SMOTE produced a more balanced outcome. The model achieved a stroke recall of 0.48 (24 out of 50 stroke cases detected) and a precision of about 0.16. Overall accuracy was 85%. Compared to the class-weighted version, the recall decreased (from 0.80 to 0.48), but the accuracy and precision both improved. This indicates that SMOTE makes Logistic Regression less aggressive in predicting stroke, trading some sensitivity (recall) for better overall performance.

**Linear SVM + SMOTE**  
The SVM model with SMOTE showed similar behaviour to Logistic Regression. It reached a stroke recall of 0.44 and a precision of about 0.15, with an overall accuracy of 85%. Like Logistic Regression, this represents a compromise between the very high recall of the class-weighted SVM and the high accuracy of the original baseline model.

**Summary of SMOTE models**  
SMOTE helps the linear models (Logistic Regression and SVM) to achieve a more balanced trade-off between recall, precision, and accuracy. However, it does not completely solve the problem of missed stroke cases. Random Forest, even with SMOTE, still misses most strokes. These results suggest that the best-performing models are the linear models (Logistic Regression and SVM) combined with either class weighting or SMOTE, and that further improvements can be obtained by adjusting the decision threshold to prioritise recall in a medical context.

## Hyperparameter Tuning

## Logistic Regression

In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10],
    'penalty': ['l2'],  # l1 removes too many features in extreme imbalance
    'solver': ['lbfgs', 'saga']
}

grid_lr = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_lr,
    scoring='recall',     # MAIN METRIC
    cv=5,
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("Best LR Params:", grid_lr.best_params_)
print("Best LR Recall:", grid_lr.best_score_)

/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nathanomenge/Desktop/Proj

Best LR Params: {'C': 1, 'penalty': 'l2', 'solver': 'saga'}
Best LR Recall: 0.828974358974359


/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



### Support Vector Machine

In [20]:
from sklearn.svm import LinearSVC

svm = LinearSVC(class_weight='balanced', random_state=42)

param_grid_svm = {
    'C': [0.001, 0.01, 0.1, 1, 10],
    'loss': ['hinge', 'squared_hinge'],
}

grid_svm = GridSearchCV(
    estimator=svm,
    param_grid=param_grid_svm,
    scoring='recall',
    cv=5,
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

print("Best SVM Params:", grid_svm.best_params_)
print("Best SVM Recall:", grid_svm.best_score_)

/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_det

Best SVM Params: {'C': 0.001, 'loss': 'hinge'}
Best SVM Recall: 0.808974358974359


/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/nathanomenge/Desktop/Projects/Learning/stroke_det

## Evaluating Tuned Models
### Linear Regression

In [25]:
best_lr = LogisticRegression(
    class_weight='balanced',
    max_iter=2000,
    random_state=42,
    C=1,
    penalty='l2',
    solver='saga'
)

best_lr.fit(X_train, y_train)
print("=== Tuned Logistic Regression (Balanced) ===")
evaluate_model(best_lr, X_test, y_test)

=== Tuned Logistic Regression (Balanced) ===
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.73      0.84       972
           1       0.13      0.82      0.23        50

    accuracy                           0.73      1022
   macro avg       0.56      0.77      0.53      1022
weighted avg       0.95      0.73      0.81      1022

ROC–AUC: 0.8339506172839506
Confusion Matrix:
[[706 266]
 [  9  41]]


/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Support Vector Machine

In [26]:
best_svm = LinearSVC(
    class_weight='balanced',
    random_state=42,
    C=0.001,
    loss='hinge'
)

best_svm.fit(X_train, y_train)
print("=== Tuned Linear SVM (Balanced) ===")
evaluate_model(best_svm, X_test, y_test)

=== Tuned Linear SVM (Balanced) ===
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.68      0.80       972
           1       0.11      0.76      0.19        50

    accuracy                           0.68      1022
   macro avg       0.55      0.72      0.50      1022
weighted avg       0.94      0.68      0.77      1022

ROC–AUC: 0.8014403292181069
Confusion Matrix:
[[662 310]
 [ 12  38]]


/Users/nathanomenge/Desktop/Projects/Learning/stroke_detection_ml/stroke_venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
